# 05 Evaluate AE Scores and Thresholds

Compute reconstruction scores for the trained convolutional autoencoder, select thresholds only on validation data, and evaluate the held-out test split with frozen thresholds.


In [1]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, precision_recall_curve
from sklearn.model_selection import StratifiedGroupKFold

import torch
import torch.nn as nn

In [2]:
REPO_ROOT = Path('..').resolve()
MANIFEST_PATH = REPO_ROOT / 'reports' / 'manifests' / 'turning_split_seed42.csv'
MODEL_PATH = REPO_ROOT / 'models' / 'ae_turning_bn16_20260711.pth'
THRESHOLD_DIR = REPO_ROOT / 'reports' / 'thresholds'
SCORE_DIR = REPO_ROOT / 'reports' / 'scores'
TABLE_DIR = REPO_ROOT / 'reports' / 'tables'
CV_DIR = REPO_ROOT / 'reports' / 'cv'
CV_MODEL_DIR = REPO_ROOT / 'models' / 'turning_ae_grouped_cv'

IMAGE_SIZE = (150, 100)
N_SVE_SEGMENTS = 5
SVE_TOP_K = 3

THRESHOLD_DIR.mkdir(parents=True, exist_ok=True)
SCORE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
CV_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
manifest = pd.read_csv(MANIFEST_PATH)
manifest = manifest[(manifest['source_dataset'] == 'turning') & manifest['split'].isin(['validation', 'test'])].copy()
manifest = manifest[manifest['image_path'].fillna('').ne('')].copy()
manifest['target'] = (manifest['label'] == 'chatter').astype(int)
manifest.groupby(['split', 'label']).size().rename('n').reset_index()

,split,label,n
0,test,chatter,27
1,test,no_chatter,48
2,validation,chatter,34
3,validation,no_chatter,70


In [4]:
def load_images(rows: pd.DataFrame, image_size=IMAGE_SIZE) -> tuple[np.ndarray, pd.DataFrame]:
    images = []
    loaded_rows = []
    for _, row in rows.iterrows():
        image_path = REPO_ROOT / row['image_path']
        if not image_path.exists():
            print(f'Missing image, skipping: {image_path}')
            continue
        image = Image.open(image_path).convert('RGB').resize(image_size)
        image = np.asarray(image, dtype='float32') / 255.0
        image = np.transpose(image, (2, 0, 1))
        images.append(image)
        loaded_rows.append(row)
    if not images:
        raise ValueError('No images loaded. Run notebooks 02 and 03 first.')
    return np.stack(images, axis=0), pd.DataFrame(loaded_rows).reset_index(drop=True)

validation_images, validation_rows = load_images(manifest[manifest['split'] == 'validation'])
test_images, test_rows = load_images(manifest[manifest['split'] == 'test'])
print('validation:', validation_images.shape)
print('test:', test_images.shape)

validation: (104, 3, 100, 150)
test: (75, 3, 100, 150)


In [5]:
def vertical_segment_scores(error_map: np.ndarray, n_segments: int, top_k: int) -> tuple[np.ndarray, np.ndarray]:
    width = error_map.shape[3]
    boundaries = np.linspace(0, width, n_segments + 1, dtype=int)
    segment_scores = []
    for left, right in zip(boundaries[:-1], boundaries[1:]):
        if right <= left:
            continue
        segment_scores.append(error_map[:, :, :, left:right].mean(axis=(1, 2, 3)))
    segment_scores = np.stack(segment_scores, axis=1)
    sve_max = segment_scores.max(axis=1)
    top_k = min(top_k, segment_scores.shape[1])
    sve_topk = np.sort(segment_scores, axis=1)[:, -top_k:].mean(axis=1)
    return sve_max, sve_topk


def reconstruct(model, images, batch_size=64):
    model.eval()
    outputs = []
    with torch.no_grad():
        for start in range(0, len(images), batch_size):
            batch = images[start:start+batch_size]
            batch = torch.tensor(
                batch,
                dtype=torch.float32,
                device=device
            )
            recon = model(batch)
            outputs.append(
                recon.cpu().numpy()
            )
    return np.concatenate(outputs, axis=0)


def score_reconstructions(model, images):
    recon = reconstruct(model, images)
    squared_error = (images - recon) ** 2
    absolute_error = np.abs(images - recon)
    sve_max, sve_topk = vertical_segment_scores(
        squared_error,
        N_SVE_SEGMENTS,
        SVE_TOP_K
    )
    return pd.DataFrame({
        'global_mse': squared_error.mean(axis=(1, 2, 3)),
        'global_mae': absolute_error.mean(axis=(1, 2, 3)),
        'sve_max': sve_max,
        'sve_topk': sve_topk,
    }) 

In [6]:
class Autoencoder(nn.Module):

    def __init__(self):
        super().__init__()
        # Encoder
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(3, 4, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(4, 8, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.MaxPool2d(kernel_size=(2, 3), stride=(2, 3)),

            nn.Conv2d(8, 12, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.flatten = nn.Flatten()
        self.encoder_fc = nn.Sequential(
            nn.Linear(12 * 25 * 25, 16),
            nn.ReLU()
        )

        # Decoder
        self.decoder_fc = nn.Sequential(
            nn.Linear(16, 12 * 25 * 25),
            nn.ReLU()
        )

        self.decoder_conv = nn.Sequential(
            nn.Conv2d(12, 12, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Upsample(scale_factor=(2, 3), mode='nearest'),

            nn.Conv2d(12, 8, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Upsample(scale_factor=(2, 2), mode='nearest'),

            nn.Conv2d(8, 4, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(4, 3, kernel_size=3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.encoder_conv(x)
        x = self.flatten(x)
        latent = self.encoder_fc(x)
        x = self.decoder_fc(latent)
        x = x.view(-1, 12, 25, 25)
        x = self.decoder_conv(x)
        return x

In [7]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
model = Autoencoder().to(device)

state_dict = torch.load(
    MODEL_PATH,
    map_location=device
)

model.load_state_dict(state_dict)

model.eval() 

Autoencoder(
  (encoder_conv): Sequential(
    (0): Conv2d(3, 4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(4, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=(2, 3), stride=(2, 3), padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(8, 12, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
  )
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (encoder_fc): Sequential(
    (0): Linear(in_features=7500, out_features=16, bias=True)
    (1): ReLU()
  )
  (decoder_fc): Sequential(
    (0): Linear(in_features=16, out_features=7500, bias=True)
    (1): ReLU()
  )
  (decoder_conv): Sequential(
    (0): Conv2d(12, 12, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Upsample(scale_factor=(2.0, 3.0), mode='nearest')
    (3): Conv2d(12, 8, kernel_size=(3, 3), stride=(1, 1), pad

In [8]:
validation_scores = pd.concat([validation_rows.reset_index(drop=True), score_reconstructions(model, validation_images)], axis=1)
test_scores = pd.concat([test_rows.reset_index(drop=True), score_reconstructions(model, test_images)], axis=1)
all_scores = pd.concat([validation_scores, test_scores], ignore_index=True)
SCORES_PATH = SCORE_DIR / 'ae_scores_turning.csv'
all_scores.to_csv(SCORES_PATH, index=False)
print(f'Wrote {SCORES_PATH}')

Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\scores\ae_scores_turning.csv


In [9]:
def select_best_f1_threshold(y_true: np.ndarray, scores: np.ndarray) -> dict:
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    if len(thresholds) == 0:
        threshold = float(np.max(scores))
        return {'threshold': threshold, 'validation_f1': 0.0, 'validation_precision': 0.0, 'validation_recall': 0.0}
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    best_idx = int(np.nanargmax(f1))
    return {
        'threshold': float(thresholds[best_idx]),
        'validation_f1': float(f1[best_idx]),
        'validation_precision': float(precision[best_idx]),
        'validation_recall': float(recall[best_idx]),
    }

def evaluate_at_threshold(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> dict:
    y_pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'pr_auc': float(average_precision_score(y_true, scores)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }

In [10]:
score_columns = ['global_mse', 'global_mae', 'sve_max', 'sve_topk']
y_val = validation_scores['target'].to_numpy()
y_test = test_scores['target'].to_numpy()
thresholds = {}
metric_rows = []

for score_name in score_columns:
    selected = select_best_f1_threshold(y_val, validation_scores[score_name].to_numpy())
    thresholds[score_name] = selected
    test_metrics = evaluate_at_threshold(y_test, test_scores[score_name].to_numpy(), selected['threshold'])
    metric_rows.append({
        'dataset': 'turning',
        'method': 'cnn_ae',
        'score': score_name,
        'threshold': selected['threshold'],
        **test_metrics,
    })

THRESHOLD_PATH = THRESHOLD_DIR / 'ae_thresholds_turning.json'
METRICS_PATH = TABLE_DIR / 'metrics_turning_ae.csv'
with THRESHOLD_PATH.open('w') as f:
    json.dump(thresholds, f, indent=2)
pd.DataFrame(metric_rows).to_csv(METRICS_PATH, index=False)
print(f'Wrote {THRESHOLD_PATH}')
print(f'Wrote {METRICS_PATH}')
pd.DataFrame(metric_rows)

Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\thresholds\ae_thresholds_turning.json
Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\tables\metrics_turning_ae.csv


,dataset,method,score,threshold,pr_auc,f1,precision,recall,tn,fp,fn,tp
0,turning,cnn_ae,global_mse,0.001864,0.974098,0.915254,0.843750,1.000000,43,5,0,27
1,turning,cnn_ae,global_mae,0.027284,0.988484,0.912281,0.866667,0.962963,44,4,1,26
2,turning,cnn_ae,sve_max,0.002774,0.960770,0.931034,0.870968,1.000000,44,4,0,27
3,turning,cnn_ae,sve_topk,0.002083,0.967683,0.900000,0.818182,1.000000,42,6,0,27


## Grouped K-Fold Cross Validation

The frozen validation/test evaluation above remains the deployment-style threshold protocol. The optional block below performs repeated stratified grouped k-fold cross validation by `source_run` so that neighboring windows from the same machining run do not leak across folds. Fold-internal best-F1 thresholds are useful for robustness analysis, while PR-AUC is the cleaner threshold-free CV summary.

In [10]:
N_CV_SPLITS = 5
CV_SEED = 42

GROUP_COLUMN = "source_run"

In [11]:
#create dataframe vor Grouped-CV

full_manifest = pd.read_csv(MANIFEST_PATH)

cv_manifest = full_manifest[
    full_manifest["source_dataset"].eq("turning")
].copy()

cv_manifest = cv_manifest[
    cv_manifest["image_path"].fillna("").ne("")
].copy()

cv_manifest["target"] = (
    cv_manifest["label"].eq("chatter")
).astype(int)

print(cv_manifest.groupby("label").size())
print("Machining runs:", cv_manifest[GROUP_COLUMN].nunique())

label
chatter        61
no_chatter    590
dtype: int64
Machining runs: 63


In [12]:
# fold declaration
cv = StratifiedGroupKFold(
    n_splits=N_CV_SPLITS,
    shuffle=True,
    random_state=CV_SEED,
)

cv_manifest["cv_fold"] = -1

X_dummy = np.zeros(len(cv_manifest))
y = cv_manifest["target"].to_numpy()
groups = cv_manifest[GROUP_COLUMN].to_numpy()

for fold, (_, test_idx) in enumerate(
    cv.split(X_dummy, y, groups),
    start=1,
):
    cv_manifest.iloc[
        test_idx,
        cv_manifest.columns.get_loc("cv_fold")
    ] = fold

fold_summary = (
    cv_manifest
    .groupby(["cv_fold", "label"])
    .size()
    .unstack(fill_value=0)
)

display(fold_summary)

label,chatter,no_chatter
cv_fold,,
1,15,121
2,11,124
3,10,118
4,11,109
5,14,118


In [13]:
# sanity check
for fold in range(1, N_CV_SPLITS + 1):

    train_groups = set(
        cv_manifest.loc[
            cv_manifest["cv_fold"] != fold,
            GROUP_COLUMN,
        ]
    )

    test_groups = set(
        cv_manifest.loc[
            cv_manifest["cv_fold"] == fold,
            GROUP_COLUMN,
        ]
    )

    assert train_groups.isdisjoint(test_groups)

In [17]:
# ============================================================
# Turning CNN-AE: Grouped 5-Fold Cross-Validation
#
# Outer split:
#   source_run groups are held out for final evaluation.
#
# Inner split:
#   remaining source_run groups are separated into
#   AE-training and threshold-calibration subsets.
#
# AE training:
#   nominal samples only.
#
# Threshold calibration:
#   labeled calibration samples.
#
# Evaluation:
#   completely held-out outer-fold machining runs.
# ============================================================

import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import GroupShuffleSplit


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

INNER_CALIBRATION_SIZE = 0.25
INNER_SEED = 42

CV_EPOCHS = 4000
CV_PATIENCE = 250
CV_BATCH_SIZE = 32
CV_LEARNING_RATE = 1e-4

SCORE_COLUMNS = [
    "global_mse",
    "global_mae",
    "sve_max",
    "sve_topk",
]

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ============================================================
# Helper: Train a fresh autoencoder
# ============================================================

def train_autoencoder(
    X_train,
    epochs=CV_EPOCHS,
    batch_size=CV_BATCH_SIZE,
    learning_rate=CV_LEARNING_RATE,
    patience=CV_PATIENCE,
):

    # Fresh weights for every outer fold
    model = Autoencoder().to(device)

    # load_images() already returns:
    # (N, C, H, W)
    train_tensor = torch.from_numpy(
        X_train
    ).float()

    train_loader = DataLoader(
        TensorDataset(
            train_tensor,
            train_tensor,
        ),
        batch_size=batch_size,
        shuffle=True,
    )

    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
    )

    best_loss = np.inf
    best_state = None
    epochs_without_improvement = 0
    loss_history = []

    for epoch in range(epochs):

        model.train()

        running_loss = 0.0
        n_samples = 0

        for inputs, targets in train_loader:

            inputs = inputs.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()

            recon = model(inputs)

            loss = criterion(
                recon,
                targets,
            )

            loss.backward()
            optimizer.step()

            current_batch_size = inputs.size(0)

            running_loss += (
                loss.item()
                * current_batch_size
            )

            n_samples += current_batch_size

        epoch_loss = (
            running_loss / n_samples
        )

        loss_history.append(
            epoch_loss
        )

        # ----------------------------------------------------
        # Early stopping based on nominal reconstruction loss
        # ----------------------------------------------------

        if epoch_loss < best_loss - 1e-8:

            best_loss = epoch_loss

            best_state = copy.deepcopy(
                model.state_dict()
            )

            epochs_without_improvement = 0

        else:

            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:

            print(
                f"Early stopping at epoch "
                f"{epoch + 1}"
            )

            break

    if best_state is not None:

        model.load_state_dict(
            best_state
        )

    model.eval()

    return model, loss_history


# ============================================================
# Helper: Calculate reconstruction scores
# ============================================================

@torch.no_grad()
def score_reconstructions_torch(
    model,
    images,
):

    tensor = (
        torch.from_numpy(images)
        .float()
        .to(device)
    )

    model.eval()

    recon = model(tensor)

    squared_error = (
        tensor - recon
    ) ** 2

    absolute_error = torch.abs(
        tensor - recon
    )

    # --------------------------------------------------------
    # Global reconstruction metrics
    # --------------------------------------------------------

    global_mse = (
        squared_error
        .mean(dim=(1, 2, 3))
        .cpu()
        .numpy()
    )

    global_mae = (
        absolute_error
        .mean(dim=(1, 2, 3))
        .cpu()
        .numpy()
    )

    # --------------------------------------------------------
    # SVE
    #
    # Expected input for vertical_segment_scores():
    # (N, C, H, W)
    # --------------------------------------------------------

    sve_max, sve_topk = (
        vertical_segment_scores(
            squared_error,
            N_SVE_SEGMENTS,
            SVE_TOP_K,
        )
    )

    # Accept both tensors and NumPy arrays
    if torch.is_tensor(sve_max):
        sve_max = (
            sve_max
            .detach()
            .cpu()
            .numpy()
        )

    if torch.is_tensor(sve_topk):
        sve_topk = (
            sve_topk
            .detach()
            .cpu()
            .numpy()
        )

    return pd.DataFrame({
        "global_mse": global_mse,
        "global_mae": global_mae,
        "sve_max": sve_max,
        "sve_topk": sve_topk,
    })


# ============================================================
# Result containers
# ============================================================

all_fold_metrics = []
all_fold_scores = []
all_fold_thresholds = []
all_fold_training = []


# ============================================================
# Outer grouped 5-fold CV
# ============================================================

for fold in range(
    1,
    N_CV_SPLITS + 1,
):

    print("\n" + "=" * 75)
    print(
        f"OUTER FOLD "
        f"{fold}/{N_CV_SPLITS}"
    )
    print("=" * 75)

    # --------------------------------------------------------
    # Outer development / evaluation split
    # --------------------------------------------------------

    outer_dev = (
        cv_manifest[
            cv_manifest["cv_fold"] != fold
        ]
        .copy()
        .reset_index(drop=True)
    )

    outer_test = (
        cv_manifest[
            cv_manifest["cv_fold"] == fold
        ]
        .copy()
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # Inner group-aware split
    #
    # No source_run can occur in both:
    #   AE training
    #   threshold calibration
    # --------------------------------------------------------

    inner_splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=INNER_CALIBRATION_SIZE,
        random_state=INNER_SEED + fold,
    )

    inner_train_idx, calibration_idx = next(
        inner_splitter.split(
            outer_dev,
            y=outer_dev["target"],
            groups=outer_dev[
                GROUP_COLUMN
            ],
        )
    )

    inner_train = (
        outer_dev
        .iloc[inner_train_idx]
        .copy()
        .reset_index(drop=True)
    )

    calibration = (
        outer_dev
        .iloc[calibration_idx]
        .copy()
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # One-class AE training:
    # only nominal samples
    # --------------------------------------------------------

    ae_train = (
        inner_train[
            inner_train["target"] == 0
        ]
        .copy()
        .reset_index(drop=True)
    )

    # ========================================================
    # Leakage / consistency checks
    # ========================================================

    train_groups = set(
        inner_train[
            GROUP_COLUMN
        ].unique()
    )

    calibration_groups = set(
        calibration[
            GROUP_COLUMN
        ].unique()
    )

    test_groups = set(
        outer_test[
            GROUP_COLUMN
        ].unique()
    )

    assert train_groups.isdisjoint(
        calibration_groups
    ), (
        f"Fold {fold}: leakage between "
        f"training and calibration groups."
    )

    assert train_groups.isdisjoint(
        test_groups
    ), (
        f"Fold {fold}: leakage between "
        f"training and evaluation groups."
    )

    assert calibration_groups.isdisjoint(
        test_groups
    ), (
        f"Fold {fold}: leakage between "
        f"calibration and evaluation groups."
    )

    assert (
        ae_train["target"] == 0
    ).all(), (
        f"Fold {fold}: AE training set "
        f"contains chatter samples."
    )

    # Threshold calibration needs both classes
    calibration_classes = set(
        calibration[
            "target"
        ].unique()
    )

    if calibration_classes != {0, 1}:

        raise ValueError(
            f"Fold {fold}: calibration set "
            f"does not contain both classes. "
            f"Found: {calibration_classes}"
        )

    # Evaluation fold must also contain both classes
    test_classes = set(
        outer_test[
            "target"
        ].unique()
    )

    if test_classes != {0, 1}:

        raise ValueError(
            f"Fold {fold}: evaluation fold "
            f"does not contain both classes. "
            f"Found: {test_classes}"
        )

    # ========================================================
    # Fold information
    # ========================================================

    print(
        f"AE training groups:      "
        f"{len(train_groups)}"
    )

    print(
        f"Calibration groups:      "
        f"{len(calibration_groups)}"
    )

    print(
        f"Evaluation groups:       "
        f"{len(test_groups)}"
    )

    print(
        f"\nAE training windows:     "
        f"{len(ae_train)} "
        f"(nominal only)"
    )

    print(
        f"Calibration windows:     "
        f"{len(calibration)}"
    )

    print(
        f"Evaluation windows:      "
        f"{len(outer_test)}"
    )

    print(
        "\nCalibration balance:"
    )

    print(
        calibration[
            "label"
        ]
        .value_counts()
        .sort_index()
    )

    print(
        "\nEvaluation balance:"
    )

    print(
        outer_test[
            "label"
        ]
        .value_counts()
        .sort_index()
    )

    # ========================================================
    # Load RGB spectrograms
    #
    # load_images already returns:
    # (N, 3, 100, 150)
    # ========================================================

    X_train, train_rows = (
        load_images(
            ae_train
        )
    )

    X_cal, cal_rows = (
        load_images(
            calibration
        )
    )

    X_test, test_rows = (
        load_images(
            outer_test
        )
    )

    print(
        "\nImage shapes:"
    )

    print(
        "AE training: ",
        X_train.shape,
    )

    print(
        "Calibration: ",
        X_cal.shape,
    )

    print(
        "Evaluation:  ",
        X_test.shape,
    )

    # ========================================================
    # Train a completely NEW CNN-AE
    # ========================================================

    model, loss_history = (
        train_autoencoder(
            X_train
        )
    )

    print(
        f"\nTraining epochs: "
        f"{len(loss_history)}"
    )

    print(
        f"Best training loss: "
        f"{min(loss_history):.6f}"
    )

    # Store training diagnostics
    all_fold_training.append({
        "outer_fold": fold,
        "epochs": len(
            loss_history
        ),
        "best_training_loss": float(
            min(loss_history)
        ),
        "n_train_windows": len(
            ae_train
        ),
        "n_calibration_windows": len(
            calibration
        ),
        "n_evaluation_windows": len(
            outer_test
        ),
        "n_train_groups": len(
            train_groups
        ),
        "n_calibration_groups": len(
            calibration_groups
        ),
        "n_evaluation_groups": len(
            test_groups
        ),
    })

    # ========================================================
    # Reconstruction scoring
    # ========================================================

    calibration_scores = (
        score_reconstructions_torch(
            model,
            X_cal,
        )
    )

    evaluation_scores = (
        score_reconstructions_torch(
            model,
            X_test,
        )
    )

    y_cal = (
        cal_rows[
            "target"
        ]
        .to_numpy()
    )

    y_test = (
        test_rows[
            "target"
        ]
        .to_numpy()
    )

    # ========================================================
    # OOF result frame
    # ========================================================

    fold_scores = (
        test_rows[
            [
                "sample_id",
                GROUP_COLUMN,
                "label",
                "target",
            ]
        ]
        .copy()
    )

    fold_scores[
        "outer_fold"
    ] = fold

    # ========================================================
    # Threshold calibration + outer-fold evaluation
    # ========================================================

    for score_name in SCORE_COLUMNS:

        # ----------------------------------------------------
        # Threshold selected ONLY on calibration data
        # ----------------------------------------------------

        selected = (
            select_best_f1_threshold(
                y_cal,
                calibration_scores[
                    score_name
                ].to_numpy(),
            )
        )

        threshold = float(
            selected[
                "threshold"
            ]
        )

        # ----------------------------------------------------
        # Final evaluation ONLY on held-out outer fold
        # ----------------------------------------------------

        current_scores = (
            evaluation_scores[
                score_name
            ]
            .to_numpy()
        )

        metrics = (
            evaluate_at_threshold(
                y_test,
                current_scores,
                threshold,
            )
        )

        # ----------------------------------------------------
        # Store fold-level metrics
        # ----------------------------------------------------

        all_fold_metrics.append({
            "dataset": "turning",
            "outer_fold": fold,
            "score": score_name,
            "threshold": threshold,
            **metrics,
        })

        # ----------------------------------------------------
        # Store calibration information
        # ----------------------------------------------------

        all_fold_thresholds.append({
            "dataset": "turning",
            "outer_fold": fold,
            "score": score_name,
            "threshold": threshold,

            "calibration_f1":
                selected[
                    "validation_f1"
                ],

            "calibration_precision":
                selected[
                    "validation_precision"
                ],

            "calibration_recall":
                selected[
                    "validation_recall"
                ],
        })

        # ----------------------------------------------------
        # Store continuous OOF score
        # ----------------------------------------------------

        fold_scores[
            score_name
        ] = current_scores

        # ----------------------------------------------------
        # Store binary OOF prediction using
        # this fold's calibrated threshold
        # ----------------------------------------------------

        fold_scores[
            f"{score_name}_prediction"
        ] = (
            current_scores
            >= threshold
        ).astype(int)

        print(
            f"\n{score_name}"
            f"\n  Threshold : "
            f"{threshold:.6f}"
            f"\n  PR-AUC    : "
            f"{metrics['pr_auc']:.4f}"
            f"\n  F1        : "
            f"{metrics['f1']:.4f}"
            f"\n  Precision : "
            f"{metrics['precision']:.4f}"
            f"\n  Recall    : "
            f"{metrics['recall']:.4f}"
        )

    all_fold_scores.append(
        fold_scores
    )

    # ========================================================
    # Release fold-specific memory
    # ========================================================

    del model
    del X_train
    del X_cal
    del X_test

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# Combine all outer-fold results
# ============================================================

cv_metrics = pd.DataFrame(
    all_fold_metrics
)

cv_thresholds = pd.DataFrame(
    all_fold_thresholds
)

cv_training = pd.DataFrame(
    all_fold_training
)

cv_scores = pd.concat(
    all_fold_scores,
    ignore_index=True,
)


# ============================================================
# Summary:
# Mean +/- standard deviation across outer folds
# ============================================================

cv_summary = (
    cv_metrics
    .groupby(
        "score"
    )
    .agg(
        pr_auc_mean=(
            "pr_auc",
            "mean",
        ),
        pr_auc_std=(
            "pr_auc",
            "std",
        ),

        f1_mean=(
            "f1",
            "mean",
        ),
        f1_std=(
            "f1",
            "std",
        ),

        precision_mean=(
            "precision",
            "mean",
        ),
        precision_std=(
            "precision",
            "std",
        ),

        recall_mean=(
            "recall",
            "mean",
        ),
        recall_std=(
            "recall",
            "std",
        ),
    )
    .reset_index()
)


print(
    "\n"
    + "=" * 75
)

print(
    "GROUPED 5-FOLD CV SUMMARY"
)

print(
    "=" * 75
)

display(
    cv_summary
)


# ============================================================
# Final sanity checks
# ============================================================

# Every sample must occur exactly once
# in the outer evaluation folds.
assert (
    cv_scores[
        "sample_id"
    ]
    .value_counts()
    .eq(1)
    .all()
), (
    "Some samples occur in more than "
    "one outer evaluation fold."
)

# Every sample in cv_manifest must have
# exactly one OOF prediction.
assert set(
    cv_scores[
        "sample_id"
    ]
) == set(
    cv_manifest[
        "sample_id"
    ]
), (
    "Not every sample received an "
    "out-of-fold prediction."
)

# Fold IDs must be valid.
assert (
    cv_scores[
        "outer_fold"
    ]
    .between(
        1,
        N_CV_SPLITS,
    )
    .all()
)

print(
    "\nFinal sanity checks passed."
)

print(
    f"OOF samples: "
    f"{len(cv_scores)}"
)

print(
    f"Unique samples: "
    f"{cv_scores['sample_id'].nunique()}"
)

print(
    f"Unique machining runs: "
    f"{cv_scores[GROUP_COLUMN].nunique()}"
)

Device: cpu

OUTER FOLD 1/5
AE training groups:      39
Calibration groups:      13
Evaluation groups:       11

AE training windows:     355 (nominal only)
Calibration windows:     128
Evaluation windows:      136

Calibration balance:
label
chatter        14
no_chatter    114
Name: count, dtype: int64

Evaluation balance:
label
chatter        15
no_chatter    121
Name: count, dtype: int64

Image shapes:
AE training:  (355, 3, 100, 150)
Calibration:  (128, 3, 100, 150)
Evaluation:   (136, 3, 100, 150)

Training epochs: 4000
Best training loss: 0.000851

global_mse
  Threshold : 0.003058
  PR-AUC    : 0.8968
  F1        : 0.8824
  Precision : 0.7895
  Recall    : 1.0000

global_mae
  Threshold : 0.031659
  PR-AUC    : 1.0000
  F1        : 0.9091
  Precision : 0.8333
  Recall    : 1.0000

sve_max
  Threshold : 0.004428
  PR-AUC    : 0.8376
  F1        : 0.8333
  Precision : 0.7143
  Recall    : 1.0000

sve_topk
  Threshold : 0.003774
  PR-AUC    : 0.8746
  F1        : 0.8333
  Precision

,score,pr_auc_mean,pr_auc_std,f1_mean,f1_std,precision_mean,precision_std,recall_mean,recall_std
0,global_mae,0.819234,0.229637,0.633542,0.266961,0.698889,0.139797,0.657403,0.387107
1,global_mse,0.787902,0.221152,0.602749,0.268569,0.633053,0.202084,0.657403,0.387107
2,sve_max,0.774750,0.215326,0.632838,0.224424,0.705395,0.222060,0.730649,0.353676
3,sve_topk,0.787553,0.219034,0.581181,0.263131,0.591917,0.222825,0.657403,0.387107



Final sanity checks passed.
OOF samples: 651
Unique samples: 651
Unique machining runs: 63


In [14]:
# Save CV-Folds 
CV_ASSIGNMENTS_PATH = (
    CV_DIR /
    "turning_ae_grouped_cv_assignments.csv"
)

cv_manifest.to_csv(
    CV_ASSIGNMENTS_PATH,
    index=False,
)